## 2.Inspect

In [64]:
import pandas as pd

YOUR_ID = 3246

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=YOUR_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1155, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           57
building_type         0
ceiling_height_m    139
price_kzt             0
dtype: int64
                   count unique         top freq        mean         std  min  \
listing_id          1155   1080   ALM-10932    3         NaN         NaN  NaN   
listed_date         1155    247  2026-04-21   10         NaN         NaN  NaN   
district            1155     48       Medeu  148         NaN         NaN  NaN   
rooms               1155      6           2  431         NaN      

In [65]:
print("rooms уникальные:", df["rooms"].unique())
print()
print("area_m2 примеры:", df["area_m2"].unique()[:20])
print()
print("price_kzt примеры:", df["price_kzt"].unique()[:20])
print()
print("district уникальные (все 48):")
print(sorted(df["district"].unique()))

rooms уникальные: ['4' '2' '3' '1' '5' 'studio']

area_m2 примеры: ['90.8' '52.1' '69.8' '82.8' '56.2' '64.3' '116.1' '56.9' '48.6' '36.6'
 '53.5' '67.6' '133.0' '59.0' '48.2' '56.3' '67.2' '110.2' '57.0' '35.4']

price_kzt примеры: ['41,560,000' '27700000' '78490000' '95690000' '30040000' '59000000'
 '63730000' '25720000' '26120000' '29170000' '29050000' '43390000'
 '177470000' '70470000' '38740000' '29 060 000 ₸' '27670000' '60140000'
 '25760000' '23580000']

district уникальные (все 48):
[' Alatau ', ' Almaly ', ' Auezov ', ' Bostandyk ', ' Medeu ', ' Nauryzbay ', ' Turksib ', ' Zhetysu ', 'ALATAU', 'ALMALY', 'AUEZOV', 'Alatau', 'Alatauskiy', 'Almalinskiy', 'Almaly', 'Auezov', 'Auezovskiy', 'BOSTANDYK', 'Bostandyk', 'Bostandykskiy', 'MEDEU', 'Medeu', 'Medeuskiy', 'NAURYZBAY', 'Nauryzbay', 'Nauryzbayskiy', 'TURKSIB', 'Turksib', 'Turksibskiy', 'ZHETYSU', 'Zhetysu', 'Zhetysuskiy', 'alatau', 'almaly', 'auezov', 'bostandyk', 'medeu', 'nauryzbay', 'turksib', 'zhetysu', 'Алатауский', 'Алма

## Data Inspection Findings

- `rooms`, `area_m2`, `price_kzt` are stored as `object` instead of numeric types.
  - `rooms` contains the string `"studio"` alongside numeric strings.
  - `price_kzt` has three formats: plain digits, comma-separated, and space-separated with a `₸` symbol.
- `listing_id`: 1155 rows but only 1080 unique values → duplicates/reposts present.
- `floor` has a minimum of -1 (impossible value, likely a sentinel).
- `year_built` has a minimum of 0 (impossible, likely a sentinel) and 57 missing values.
- `ceiling_height_m` has 139 missing values.
- `district` has 48 raw string variants (casing, whitespace, transliteration suffix `-skiy`, Russian names) that map down to 8 canonical districts: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu.

In [66]:
validation_spec = {
    "listing_id":       {"dtype": "string", "notes": "unique per real listing, but duplicates/reposts exist"},
    "listed_date":      {"dtype": "datetime"},
    "district":         {"dtype": "category", "allowed": ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]},
    "rooms":            {"dtype": "int", "min": 0, "max": 10, "notes": "'studio' -> 0"},
    "area_m2":          {"dtype": "float", "min": 10, "max": 500},
    "floor":            {"dtype": "int", "min": 1, "max": 60, "notes": "current min=-1 is impossible"},
    "total_floors":     {"dtype": "int", "min": 1, "max": 60},
    "year_built":       {"dtype": "int", "min": 1950, "max": 2026, "notes": "current min=0 is impossible/sentinel"},
    "building_type":    {"dtype": "category"},
    "ceiling_height_m": {"dtype": "float", "min": 2.0, "max": 4.5},
    "price_kzt":        {"dtype": "float", "min": 1_000_000, "max": 500_000_000},
}
validation_spec

{'listing_id': {'dtype': 'string',
  'notes': 'unique per real listing, but duplicates/reposts exist'},
 'listed_date': {'dtype': 'datetime'},
 'district': {'dtype': 'category',
  'allowed': ['Alatau',
   'Almaly',
   'Auezov',
   'Bostandyk',
   'Medeu',
   'Nauryzbay',
   'Turksib',
   'Zhetysu']},
 'rooms': {'dtype': 'int', 'min': 0, 'max': 10, 'notes': "'studio' -> 0"},
 'area_m2': {'dtype': 'float', 'min': 10, 'max': 500},
 'floor': {'dtype': 'int',
  'min': 1,
  'max': 60,
  'notes': 'current min=-1 is impossible'},
 'total_floors': {'dtype': 'int', 'min': 1, 'max': 60},
 'year_built': {'dtype': 'int',
  'min': 1950,
  'max': 2026,
  'notes': 'current min=0 is impossible/sentinel'},
 'building_type': {'dtype': 'category'},
 'ceiling_height_m': {'dtype': 'float', 'min': 2.0, 'max': 4.5},
 'price_kzt': {'dtype': 'float', 'min': 1000000, 'max': 500000000}}

## 3.Clean

In [67]:
dup_ids = df["listing_id"][df["listing_id"].duplicated(keep=False)]
dup_rows = df[df["listing_id"].isin(dup_ids)].sort_values("listing_id")
print(dup_rows.shape)
dup_rows.head(20)

(147, 11)


,listing_id,listed_date,district,rooms,area_m2,floor,total_floors,year_built,building_type,ceiling_height_m,price_kzt
8,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
962,ALM-10003,2026-05-04,Auezov,2,48.6,2,5,1978.0,panel,2.55,28320000
819,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
935,ALM-10009,2025-12-03,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,42700000
179,ALM-10009,2026-01-19,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,39400000
453,ALM-10016,2026-01-29,Auezov,1,39.7,13,25,2022.0,monolith,2.80,32100000
477,ALM-10016,2026-01-20,Auezov,1,39.7,13,25,2022.0,monolith,2.80,35130000
255,ALM-10021,2026-04-06,Alatau,3,71.3,9,12,1988.0,panel,2.69,32300000
609,ALM-10021,2026-05-09,Alatau,3,71.3,9,12,1988.0,panel,2.69,30090000
52,ALM-10056,2026-02-08,Bostandyk,4,107.2,9,9,2006.0,monolith,2.81,101060000


In [68]:
before = len(df)

df["listed_date"] = pd.to_datetime(df["listed_date"])

df_clean = (
    df.sort_values("listed_date")
      .drop_duplicates(subset="listing_id", keep="last")
      .reset_index(drop=True)
)

after = len(df_clean)
print(f"Rows before: {before}, after dedup: {after}, removed: {before - after}")

Rows before: 1155, after dedup: 1080, removed: 75


In [69]:
canonical_districts = ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]

ru_map = {
    "Алатауский": "Alatau",
    "Алмалинский": "Almaly",
    "Ауэзовский": "Auezov",
    "Бостандыкский": "Bostandyk",
    "Медеуский": "Medeu",
    "Наурызбайский": "Nauryzbay",
    "Турксибский": "Turksib",
    "Жетысуский": "Zhetysu",
}

def clean_district(x):
    x = x.strip()
    if x in ru_map:
        return ru_map[x]
    x = x.lower()
    for suffix in ["inskiy", "skiy"]:
        if x.endswith(suffix):
            x = x[: -len(suffix)]
            break
    for d in canonical_districts:
        if d.lower().startswith(x):
            return d
    return None

df_clean["district"] = df_clean["district"].apply(clean_district)

print(df_clean["district"].unique())
print(df_clean["district"].isna().sum())

['Almaly' 'Zhetysu' 'Alatau' 'Medeu' 'Bostandyk' 'Auezov' 'Turksib'
 'Nauryzbay']
0


Districts normalized to 8 canonical values by stripping whitespace,lowercasing,mapping Russian names via a dictionary and stripping transliteration suffixes (`-skiy`/`-inskiy`). `Almalinskiy` required a `startswith` match rather than exact match due to an irregular suffix pattern

In [70]:
# rooms: 'studio' в 0,остальное в число
df_clean["rooms"] = df_clean["rooms"].replace("studio", "0").astype(int)

# area_m2: убрать "m²",меняем запятую на точку,приведем к float
df_clean["area_m2"] = (
    df_clean["area_m2"]
    .astype(str)
    .str.replace("m²", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

# price_kzt: убираем запятые,пробелы,символ ₸
df_clean["price_kzt"] = (
    df_clean["price_kzt"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₸", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
    .astype(float)
)

print(df_clean[["rooms", "area_m2", "price_kzt"]].dtypes)
print(df_clean[["rooms", "area_m2", "price_kzt"]].describe())

rooms          int64
area_m2      float64
price_kzt    float64
dtype: object
             rooms      area_m2     price_kzt
count  1080.000000  1080.000000  1.080000e+03
mean      2.236111    68.907963  4.631151e+07
std       1.051409    59.780981  2.521701e+07
min       0.000000    25.300000  1.144000e+07
25%       1.000000    45.000000  2.828750e+07
50%       2.000000    58.750000  4.019000e+07
75%       3.000000    78.650000  5.732500e+07
max       5.000000   879.000000  1.930000e+08
